In [2]:
import pandas as pd 
import os 
from sqlalchemy import create_engine

In [2]:
engine = create_engine('sqlite:///inventory.db')

In [18]:
for file in os.listdir('C:/Users/KAUSHAL/Downloads/data1'):
    print(file)

begin_inventory.csv
end_inventory.csv
purchases.csv
purchase_prices.csv
sales.csv
vendor_invoice.csv


In [19]:
import warnings
warnings.filterwarnings('ignore')

path = 'C:/Users/KAUSHAL/Downloads/data1/'

vendor_invoice  = pd.read_csv(path + 'vendor_invoice.csv')
purchase_prices = pd.read_csv(path + 'purchase_prices.csv')
purchases       = pd.read_csv(path + 'purchases.csv', nrows=500000)
begin_inv       = pd.read_csv(path + 'begin_inventory.csv')
end_inv         = pd.read_csv(path + 'end_inventory.csv')

print("✅ vendor_invoice :", vendor_invoice.shape)
print("✅ purchase_prices:", purchase_prices.shape)
print("✅ purchases      :", purchases.shape)
print("✅ begin_inventory:", begin_inv.shape)
print("✅ end_inventory  :", end_inv.shape)

✅ vendor_invoice : (5543, 10)
✅ purchase_prices: (12261, 9)
✅ purchases      : (500000, 16)
✅ begin_inventory: (206529, 9)
✅ end_inventory  : (224489, 9)


In [20]:
import sqlite3

conn = sqlite3.connect('C:/Users/KAUSHAL/Downloads/data1/vendor_performance.db')

vendor_invoice.to_sql('vendor_invoice',   conn, if_exists='replace', index=False)
purchase_prices.to_sql('purchase_prices', conn, if_exists='replace', index=False)
purchases.to_sql('purchases',             conn, if_exists='replace', index=False)
begin_inv.to_sql('begin_inventory',       conn, if_exists='replace', index=False)
end_inv.to_sql('end_inventory',           conn, if_exists='replace', index=False)

print("✅ All tables loaded into SQLite!")

✅ All tables loaded into SQLite!


In [21]:
print("=== VENDOR INVOICE ===")
print(vendor_invoice.columns.tolist())

print("\n=== PURCHASE PRICES ===")
print(purchase_prices.columns.tolist())

print("\n=== PURCHASES ===")
print(purchases.columns.tolist())

print("\n=== BEGIN INVENTORY ===")
print(begin_inv.columns.tolist())

=== VENDOR INVOICE ===
['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate', 'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval']

=== PURCHASE PRICES ===
['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification', 'PurchasePrice', 'VendorNumber', 'VendorName']

=== PURCHASES ===
['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber', 'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate', 'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification']

=== BEGIN INVENTORY ===
['InventoryId', 'Store', 'City', 'Brand', 'Description', 'Size', 'onHand', 'Price', 'startDate']


In [22]:
# Query 1 - Total Purchase & Vendor Summary
vendor_summary = pd.read_sql_query("""
    SELECT 
        VendorName,
        VendorNumber,
        COUNT(DISTINCT PONumber) as TotalOrders,
        SUM(Quantity) as TotalPurchaseQuantity,
        SUM(Dollars) as TotalPurchaseDollars,
        SUM(Freight) as TotalFreight,
        ROUND(SUM(Dollars) / COUNT(DISTINCT PONumber), 2) as AvgOrderValue
    FROM vendor_invoice
    GROUP BY VendorName, VendorNumber
    ORDER BY TotalPurchaseDollars DESC
""", conn)

print("Vendor Summary (Top 10):")
print(vendor_summary.head(10))

Vendor Summary (Top 10):
                    VendorName  VendorNumber  TotalOrders  \
0  DIAGEO NORTH AMERICA INC             3960           55   
1        MARTIGNETTI COMPANIES          4425           55   
2  JIM BEAM BRANDS COMPANY             12546           55   
3  PERNOD RICARD USA                   17035           55   
4  BACARDI USA INC                       480           55   
5  CONSTELLATION BRANDS INC             1392           55   
6  BROWN-FORMAN CORP                    1128           55   
7  ULTRA BEVERAGE COMPANY LLP           9165           55   
8  E & J GALLO WINERY                   3252           55   
9  M S WALKER INC                       9552           55   

   TotalPurchaseQuantity  TotalPurchaseDollars  TotalFreight  AvgOrderValue  
0                5459788           50959796.85     257032.07      926541.76  
1                2637275           27821473.91     144719.92      505844.98  
2                2737165           24203151.05     123880.97      440

In [23]:
# Query 2 - Purchase Price Analysis by Vendor
price_analysis = pd.read_sql_query("""
    SELECT 
        VendorName,
        VendorNumber,
        COUNT(DISTINCT Brand) as TotalBrands,
        ROUND(AVG(PurchasePrice), 2) as AvgPurchasePrice,
        SUM(Quantity) as TotalQtyPurchased,
        SUM(Dollars) as TotalSpend
    FROM purchases
    GROUP BY VendorName, VendorNumber
    ORDER BY TotalSpend DESC
    LIMIT 20
""", conn)

print("Price Analysis (Top 10):")
print(price_analysis.head(10))

Price Analysis (Top 10):
                    VendorName  VendorNumber  TotalBrands  AvgPurchasePrice  \
0  DIAGEO NORTH AMERICA INC             3960          323             14.91   
1  JIM BEAM BRANDS COMPANY             12546          308             12.55   
2        MARTIGNETTI COMPANIES          4425          906             10.80   
3  PERNOD RICARD USA                   17035          213             18.65   
4  CONSTELLATION BRANDS INC             1392          353              6.91   
5  BROWN-FORMAN CORP                    1128           98             17.76   
6  ULTRA BEVERAGE COMPANY LLP           9165          529             15.88   
7  BACARDI USA INC                       480          139             13.55   
8  E & J GALLO WINERY                   3252          383              7.28   
9  M S WALKER INC                       9552          533              8.34   

   TotalQtyPurchased  TotalSpend  
0            1143584  9646484.40  
1             637360  5313231.86  


In [24]:
# Query 3 - Brand Performance
brand_performance = pd.read_sql_query("""
    SELECT 
        p.Brand,
        p.Description,
        p.VendorName,
        pp.Price as RetailPrice,
        pp.PurchasePrice,
        ROUND(pp.Price - pp.PurchasePrice, 2) as GrossProfit,
        ROUND((pp.Price - pp.PurchasePrice) / pp.Price * 100, 2) as ProfitMargin,
        SUM(p.Quantity) as TotalQty,
        SUM(p.Dollars) as TotalPurchase,
        ROUND(SUM(p.Quantity) * pp.Price, 2) as EstimatedSales
    FROM purchases p
    JOIN purchase_prices pp 
        ON p.Brand = pp.Brand AND p.VendorNumber = pp.VendorNumber
    GROUP BY p.Brand, p.VendorName
    ORDER BY EstimatedSales DESC
    LIMIT 20
""", conn)

print("Brand Performance (Top 10):")
print(brand_performance.head(10))

Brand Performance (Top 10):
   Brand              Description                   VendorName  RetailPrice  \
0   4261   Capt Morgan Spiced Rum  DIAGEO NORTH AMERICA INC           22.99   
1   1233  Jack Daniels No 7 Black  BROWN-FORMAN CORP                  36.99   
2   3545          Ketel One Vodka  DIAGEO NORTH AMERICA INC           29.99   
3   3405    Tito's Handmade Vodka        MARTIGNETTI COMPANIES        28.99   
4   8068         Absolut 80 Proof  PERNOD RICARD USA                  24.99   
5   3102        Smirnoff Traveler  DIAGEO NORTH AMERICA INC           17.99   
6   2589    Jameson Irish Whiskey  PERNOD RICARD USA                  39.99   
7   2585    Jameson Irish Whiskey  PERNOD RICARD USA                  22.99   
8   3858         Grey Goose Vodka  BACARDI USA INC                    23.99   
9   3802       Three Olives Vodka  PROXIMO SPIRITS INC.               18.99   

   PurchasePrice  GrossProfit  ProfitMargin  TotalQty  TotalPurchase  \
0          16.17         6.82 

In [25]:
# Query 4 - Unsold Capital (Begin - End Inventory)
unsold_capital = pd.read_sql_query("""
    SELECT 
        b.Brand,
        b.Description,
        b.City,
        b.onHand as BeginStock,
        e.onHand as EndStock,
        (b.onHand - e.onHand) as StockSold,
        b.Price as RetailPrice,
        ROUND((b.onHand - e.onHand) * b.Price, 2) as EstimatedRevenue,
        ROUND(e.onHand * b.Price, 2) as UnsoldCapital
    FROM begin_inventory b
    JOIN end_inventory e 
        ON b.InventoryId = e.InventoryId
    WHERE e.onHand > 0
    ORDER BY UnsoldCapital DESC
    LIMIT 20
""", conn)

print("Unsold Capital (Top 10):")
print(unsold_capital.head(10))

Unsold Capital (Top 10):
   Brand                 Description       City  BeginStock  EndStock  \
0   1233     Jack Daniels No 7 Black  MOUNTMEND         537      1246   
1   2753  Johnnie Walker Black Label  MOUNTMEND         579       650   
2   3545             Ketel One Vodka  MOUNTMEND         515      1176   
3   3405       Tito's Handmade Vodka  MOUNTMEND         339      1164   
4   8068            Absolut 80 Proof  MOUNTMEND         601      1273   
5   2589       Jameson Irish Whiskey  MOUNTMEND         167       782   
6   2757    Johnnie Walker Red Label  MOUNTMEND         389      1003   
7   3405       Tito's Handmade Vodka  PITMERDEN         367       969   
8   2753  Johnnie Walker Black Label  MOUNTMEND         266       442   
9   3545             Ketel One Vodka  PITMERDEN         711       853   

   StockSold  RetailPrice  EstimatedRevenue  UnsoldCapital  
0       -709        35.99         -25516.91       44843.54  
1        -71        61.99          -4401.29      

In [26]:
# Query 5 - Low Performing Vendors
low_performers = pd.read_sql_query("""
    SELECT 
        p.VendorName,
        pp.PurchasePrice,
        pp.Price as RetailPrice,
        ROUND((pp.Price - pp.PurchasePrice) / pp.Price, 3) as ProfitMargin,
        SUM(p.Quantity) as TotalQty,
        SUM(p.Dollars) as TotalSpend
    FROM purchases p
    JOIN purchase_prices pp 
        ON p.Brand = pp.Brand AND p.VendorNumber = pp.VendorNumber
    GROUP BY p.VendorName
    HAVING ProfitMargin < 0.8
    ORDER BY ProfitMargin ASC
    LIMIT 10
""", conn)

print("Low Performing Vendors:")
print(low_performers)

Low Performing Vendors:
                    VendorName  PurchasePrice  RetailPrice  ProfitMargin  \
0  MARSALLE COMPANY                     31.99        39.99         0.200   
1  SIDNEY FRANK IMPORTING CO            30.39        37.99         0.200   
2  CALEDONIA SPIRITS INC                29.36        36.99         0.206   
3  DUGGANS DISTILLED PRODUCTS           17.45        21.99         0.206   
4  DIAGEO NORTH AMERICA INC             13.48        16.99         0.207   
5  SEA HAGG DISTILLERY LLC              22.21        27.99         0.207   
6  TALL SHIP DISTILLERY LLC             22.21        27.99         0.207   
7  FLAG HILL WINERY & VINEYARD          13.38        16.99         0.212   
8  HEAVEN HILL DISTILLERIES              8.26        10.49         0.213   
9  PREMIER DISTRIBUTORS                 16.01        20.49         0.219   

   TotalQty  TotalSpend  
0      1405    37550.33  
1     44800   373669.85  
2       823    25187.57  
3      1446    24302.78  
4   11435

In [27]:
# Query 6 - Purchase Contribution %
purchase_contribution = pd.read_sql_query("""
    SELECT 
        VendorName,
        SUM(Dollars) as TotalPurchase,
        ROUND(SUM(Dollars) * 100.0 / (SELECT SUM(Dollars) FROM vendor_invoice), 2) as ContributionPct
    FROM vendor_invoice
    GROUP BY VendorName
    ORDER BY ContributionPct DESC
    LIMIT 10
""", conn)

print("Purchase Contribution %:")
print(purchase_contribution)

Purchase Contribution %:
                    VendorName  TotalPurchase  ContributionPct
0  DIAGEO NORTH AMERICA INC       50959796.85            15.83
1        MARTIGNETTI COMPANIES    27821473.91             8.64
2  JIM BEAM BRANDS COMPANY        24203151.05             7.52
3  PERNOD RICARD USA              24124091.56             7.49
4  BACARDI USA INC                17624378.72             5.48
5  CONSTELLATION BRANDS INC       15573917.90             4.84
6  BROWN-FORMAN CORP              13529433.08             4.20
7  ULTRA BEVERAGE COMPANY LLP     13210613.93             4.10
8  E & J GALLO WINERY             12289608.09             3.82
9  M S WALKER INC                 10935817.30             3.40


In [28]:
vendor_summary.to_csv('C:/Users/KAUSHAL/Downloads/data1/vendor_summary.csv', index=False)
price_analysis.to_csv('C:/Users/KAUSHAL/Downloads/data1/price_analysis.csv', index=False)
brand_performance.to_csv('C:/Users/KAUSHAL/Downloads/data1/brand_performance.csv', index=False)
unsold_capital.to_csv('C:/Users/KAUSHAL/Downloads/data1/unsold_capital.csv', index=False)
low_performers.to_csv('C:/Users/KAUSHAL/Downloads/data1/low_performers.csv', index=False)
purchase_contribution.to_csv('C:/Users/KAUSHAL/Downloads/data1/purchase_contribution.csv', index=False)

print("✅ All result files saved!")
print("📁 Check your data1 folder — 6 new CSV files created!")

✅ All result files saved!
📁 Check your data1 folder — 6 new CSV files created!


In [3]:
sales = pd.read_csv('C:/Users/KAUSHAL/Downloads/data1/sales.csv', nrows=3)
print(sales.columns.tolist())

['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity', 'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification', 'ExciseTax', 'VendorNo', 'VendorName']


In [5]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('C:/Users/KAUSHAL/Downloads/data1/vendor_performance.db')
print("Connected ✅")

Connected ✅


In [6]:
query = """
    SELECT 
        VendorName,
        SUM(Dollars) as TotalPurchase,
        COUNT(*) as TotalOrders,
        ROUND(SUM(Dollars) / (SELECT SUM(Dollars) FROM vendor_invoice) * 100, 4) as PurchaseSharePct
    FROM vendor_invoice
    GROUP BY VendorName
    ORDER BY PurchaseSharePct ASC
    LIMIT 15
"""
result = pd.read_sql_query(query, conn)
print(result)
print("\nMin:", result['PurchaseSharePct'].min())
print("Max:", result['PurchaseSharePct'].max())

                     VendorName  TotalPurchase  TotalOrders  PurchaseSharePct
0   AAPER ALCOHOL & CHEMICAL CO         105.07            1            0.0000
1   CAPSTONE INTERNATIONAL               54.64            1            0.0000
2   FANTASY FINE WINES CORP             128.64            2            0.0000
3   FLAVOR ESSENCE INC                   17.00            1            0.0000
4   LAUREATE IMPORTS CO                 140.94            1            0.0000
5   SILVER MOUNTAIN CIDERS               77.18            2            0.0000
6   TRUETT HURST                        236.64            1            0.0001
7   AMERICAN SPIRITS EXCHANGE          1205.16            6            0.0004
8   BLACK ROCK SPIRITS LLC             1152.10           18            0.0004
9   BRONCO WINE COMPANY                2071.90           26            0.0006
10  MILTONS DISTRIBUTING CO            1824.18           10            0.0006
11  APPOLO VINEYARDS LLC               2399.70           13     